# Hazelnut Price Models

**Target:** annual crop-year USD/kg log-return (`ret_vwap_usd`)  
**Primary driver:** production shortfall vs trailing 5yr trend  

| Model | Spec | R² | n | Notes |
|-------|------|----|---|-------|
| **M1** | `ret_usd ~ shortfall_trail` | 0.529 | 22 | Primary — clean, no look-ahead |
| **M2** | `ret_usd ~ shortfall_pct` | 0.557 | 22 | Centered trend — look-ahead upper bound |
| **M3** | `ret_usd ~ shortfall_trail + log(tmo)` | tbd | 12 | Adds TMO floor; subsample only |
| **M4** | `log(USD) ~ shortfall_pct + month + month²` | 0.189 | ~280 | Monthly honest signal |

Rejected: demand-side equities (Barry, Hershey, Bunge, cocoa), frost degree-hours, prod_ret_lag1 — none significant after controlling for shortfall.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from pathlib import Path

plt.style.use('seaborn-v0_8-whitegrid')
DATA = Path('../data/raw')

In [ ]:
# --- Load ---
master  = pd.read_csv(DATA / 'hazelnut_35yr_master.csv', index_col=0)
master.index = master.index.astype(int)

monthly = pd.read_csv(DATA / 'giresun_spot_prices_monthly.csv')
cy      = pd.read_csv(DATA / 'giresun_spot_prices_cropyear.csv').set_index('crop_year')

# --- Trailing 5yr shortfall (no look-ahead) ---
master['trend_trail5']    = master['prod_mt'].rolling(5, min_periods=3).mean().shift(1)
master['shortfall_trail'] = (master['prod_mt'] - master['trend_trail5']) / master['trend_trail5'] * 100

# --- Annual panel ---
prod = master[['prod_mt', 'prod_shortfall', 'shortfall_trail', 'tmo_try']].copy()
prod.index.name = 'crop_year'
annual = cy.join(prod, how='left')
annual['log_usd']       = np.log(annual['vwap_usd_kg_inshell'])
annual['ret_usd']       = annual['log_usd'].diff()
annual['shortfall_pct'] = annual['prod_shortfall'] * 100
annual['log_tmo']       = np.log(annual['tmo_try'].replace(0, np.nan))

# --- Monthly panel ---
df = monthly.merge(prod.reset_index(), on='crop_year', how='left')
df['crop_month']    = df.groupby('crop_year').cumcount() + 1
df['log_usd']       = np.log(df['avg_usd_kg_inshell'].replace(0, np.nan))
df['shortfall_pct'] = df['prod_shortfall'] * 100

print(f'Annual: {len(annual)} years   Monthly: {len(df)} rows ({df.crop_year.nunique()} crop years)')

## M1 — Annual USD return ~ trailing shortfall  *(primary)*

In [ ]:
a1 = annual.dropna(subset=['ret_usd', 'shortfall_trail'])
m1 = sm.OLS(a1['ret_usd'], sm.add_constant(a1[['shortfall_trail']])).fit()
print(m1.summary2())

## M2 — Annual USD return ~ centered shortfall  *(look-ahead upper bound)*

In [ ]:
a2 = annual.dropna(subset=['ret_usd', 'shortfall_pct'])
m2 = sm.OLS(a2['ret_usd'], sm.add_constant(a2[['shortfall_pct']])).fit()
print(m2.summary2())

## M3 — Annual USD return ~ trailing shortfall + TMO floor  *(n=12 subsample)*

`log(tmo_try)` is the announced floor price — exogenous policy variable, not endogenous.

In [ ]:
a3 = annual.dropna(subset=['ret_usd', 'shortfall_trail', 'log_tmo'])
print(f'n = {len(a3)} crop years: {list(a3.index)}')
m3 = sm.OLS(a3['ret_usd'], sm.add_constant(a3[['shortfall_trail', 'log_tmo']])).fit()
print(m3.summary2())

## M4 — Monthly log(USD) ~ shortfall + seasonality  *(honest monthly signal)*

No lagged price — R² reflects production signal only, not AR persistence.

In [ ]:
d4 = df.dropna(subset=['log_usd', 'shortfall_pct']).copy()
d4['crop_month_sq'] = d4['crop_month'] ** 2
m4 = sm.OLS(d4['log_usd'], sm.add_constant(d4[['shortfall_pct', 'crop_month', 'crop_month_sq']])).fit(
    cov_type='cluster', cov_kwds={'groups': d4['crop_year']})
print(m4.summary2())

## Comparison

In [ ]:
rows = [
    ('M1', 'ret_usd ~ shortfall_trail',              m1, 'shortfall_trail', 'annual',  'primary — clean'),
    ('M2', 'ret_usd ~ shortfall_pct (centered)',     m2, 'shortfall_pct',   'annual',  'look-ahead upper bound'),
    ('M3', 'ret_usd ~ shortfall_trail + log(tmo)',   m3, 'shortfall_trail', 'annual',  'n=12 TMO subsample'),
    ('M4', 'log(USD) ~ shortfall + month + month²',  m4, 'shortfall_pct',   'monthly', 'honest, no lag'),
]

cmp = pd.DataFrame([
    {'': tag, 'Spec': spec, 'Freq': freq,
     'R²':   round(m.rsquared, 3),
     'AIC':  round(m.aic, 1),
     'n':    int(m.nobs),
     'shortfall coef': round(m.params[feat], 4),
     'p':    round(m.pvalues[feat], 3),
     'Note': note}
    for tag, spec, m, feat, freq, note in rows
]).set_index('')
cmp

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# M1: scatter + fit
ax = axes[0]
ax.scatter(a1['shortfall_trail'], a1['ret_usd'] * 100, s=40, c='steelblue', zorder=3)
xs = np.linspace(a1['shortfall_trail'].min(), a1['shortfall_trail'].max(), 100)
p  = m1.params
ax.plot(xs, (p['const'] + p['shortfall_trail'] * xs) * 100, 'k--', lw=1.2)
for yr, row in a1.iterrows():
    if abs(row['shortfall_trail']) > 15 or abs(row['ret_usd']) > 0.35:
        ax.annotate(str(yr), (row['shortfall_trail'], row['ret_usd']*100),
                    xytext=(4,3), textcoords='offset points', fontsize=7)
ax.axhline(0, color='grey', lw=0.5, ls='--'); ax.axvline(0, color='grey', lw=0.5, ls='--')
ax.set_xlabel('Shortfall vs trailing 5yr trend (%)'); ax.set_ylabel('Annual USD return (%)')
ax.set_title(f'M1  R²={m1.rsquared:.3f}')

# M1 vs M2: actual vs fitted overlay
ax = axes[1]
fit1 = m1.fittedvalues * 100
fit2 = m2.fittedvalues * 100
act  = a1['ret_usd'] * 100
ax.scatter(fit1, act, s=30, c='steelblue', label=f'M1 trailing (R²={m1.rsquared:.3f})', zorder=3)
ax.scatter(fit2, a2['ret_usd']*100, s=30, c='#e74c3c', marker='x',
           label=f'M2 centered (R²={m2.rsquared:.3f})', zorder=3)
mn, mx = min(fit1.min(), fit2.min(), act.min())-2, max(fit1.max(), fit2.max(), act.max())+2
ax.plot([mn,mx],[mn,mx],'k--',lw=0.8)
ax.set_xlabel('Fitted (%)'); ax.set_ylabel('Actual (%)')
ax.set_title('M1 vs M2 — actual vs fitted')
ax.legend(fontsize=8)

# M4 monthly: residuals vs shortfall
ax = axes[2]
resid4 = d4['log_usd'] - m4.fittedvalues
ax.scatter(d4['shortfall_pct'], resid4, s=5, alpha=0.4, c='steelblue')
ax.axhline(0, color='grey', lw=0.5)
ax.set_xlabel('Production shortfall (%)')
ax.set_ylabel('Residual log(USD)')
ax.set_title(f'M4 monthly residuals  R²={m4.rsquared:.3f}')

plt.tight_layout()
plt.show()